# 06 오답 분석 및 최종 모델 선택

05번 결과를 바탕으로 최종 모델을 선택하고, 선택 모델 (Metadata only + MLP)의 오답 데이터를 분석한다.

현재 05번 실행 결과에서는 선택 모델 후보 (Metadata only + MLP)가 test 기준 F1, PR-AUC, Recall에서 가장 좋은 성능을 보였다. 따라서 06번에서는 이 결과를 숨기지 않고, 왜 메타데이터만 사용한 모델이 가장 좋았는지와 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)이 기대보다 낮았던 이유를 함께 분석한다.

06번은 별점 정제 단계가 아니다. 별점 정제는 여기서 저장한 선택 모델 (Metadata only + MLP)을 07번에서 불러와 진행한다.


## 1. 라이브러리 로드

저장된 모델을 불러오고, test split 기준 성능과 오답 데이터를 다시 계산한다.


In [1]:
import os

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split


## 2. test 데이터와 원본 리뷰 데이터 로드

05번과 같은 split을 재현해서 `final_hybrid_test.csv`의 각 행이 원본 리뷰의 어떤 행인지 연결한다.

이 연결이 되어야 FP/FN 오답 리뷰의 원문, 별점, 사진 수, 텍스트 길이 등을 같이 볼 수 있다.


In [2]:
SEED = 42
TARGET_COL = 'target_label'
LABEL_COL = 'label'
TEXT_COL = 'cleaned_review_text'
META_COLS = ['meta_rating', 'meta_text_length', 'meta_emoji_count', 'meta_photo_count']
RAW_META_COLS = ['rating', 'text_length', 'emoji_count', 'photo_count']

train_df = pd.read_csv('csv/final_hybrid_train.csv')
val_df = pd.read_csv('csv/final_hybrid_val.csv')
test_df = pd.read_csv('csv/final_hybrid_test.csv')
raw_df = pd.read_csv('csv/preprocessed_reviews.csv')

pca_cols = [col for col in train_df.columns if col.startswith('feat_')]
meta_cols = META_COLS
hybrid_cols = pca_cols + meta_cols

y_test = test_df[TARGET_COL].astype(int)

train_idx, temp_idx = train_test_split(
    raw_df.index,
    test_size=0.3,
    random_state=SEED,
    stratify=raw_df[LABEL_COL],
)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    random_state=SEED,
    stratify=raw_df.loc[temp_idx, LABEL_COL],
)

np.testing.assert_array_equal(raw_df.loc[test_idx, LABEL_COL].astype(int).to_numpy(), y_test.to_numpy())

test_review_df = raw_df.loc[test_idx].copy().reset_index().rename(columns={'index': 'raw_index'})
text_test = test_review_df[TEXT_COL].fillna('').astype(str)

print('test hybrid shape:', test_df.shape)
print('test review shape:', test_review_df.shape)
print('test label 분포:', y_test.value_counts().sort_index().to_dict())


test hybrid shape: (1326, 148)
test review shape: (1326, 16)
test label 분포: {0: 853, 1: 473}


## 3. 저장된 모델 bundle 로드

05번에서 저장한 baseline/ablation 모델과 04번에서 저장한 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)을 모두 불러온다.


In [3]:
model_specs = [
    {
        'model': 'baseline_tfidf_random_forest',
        'feature_set': 'cleaned_text_tfidf',
        'path': 'outputs/baseline_tfidf_random_forest_model.joblib',
        'input_type': 'text',
    },
    {
        'model': 'ablation_mlp_kcbert_pca_only',
        'feature_set': 'kcbert_pca_only',
        'path': 'outputs/ablation_mlp_kcbert_pca_only_model.joblib',
        'input_type': 'tabular',
    },
    {
        'model': 'ablation_mlp_metadata_only',
        'feature_set': 'metadata_only',
        'path': 'outputs/ablation_mlp_metadata_only_model.joblib',
        'input_type': 'tabular',
    },
    {
        'model': 'proposed_hybrid_mlp_04',
        'feature_set': 'kcbert_pca+metadata',
        'path': 'outputs/proposed_mlp_final_model.joblib',
        'input_type': 'tabular',
    },
]

missing_paths = [spec['path'] for spec in model_specs if not os.path.exists(spec['path'])]
if missing_paths:
    raise FileNotFoundError(f'모델 파일이 없습니다: {missing_paths}')

model_bundles = {spec['model']: joblib.load(spec['path']) for spec in model_specs}

loaded_models = pd.DataFrame([
    {'model': spec['model'], 'feature_set': spec['feature_set'], 'path': spec['path']}
    for spec in model_specs
])
display(loaded_models)


,model,feature_set,path
0,baseline_tfidf_random_forest,cleaned_text_tfidf,outputs/baseline_tfidf_random_forest_model.joblib
1,ablation_mlp_kcbert_pca_only,kcbert_pca_only,outputs/ablation_mlp_kcbert_pca_only_model.joblib
2,ablation_mlp_metadata_only,metadata_only,outputs/ablation_mlp_metadata_only_model.joblib
3,proposed_hybrid_mlp_04,kcbert_pca+metadata,outputs/proposed_mlp_final_model.joblib


## 4. 모델별 test 성능 비교

저장된 모델을 모두 같은 test 데이터에 다시 적용한다.

최종 비교는 `best_threshold` 기준을 사용하되, 각 모델 bundle 안에 저장된 threshold를 그대로 사용한다.


In [4]:
def predict_prob(spec, bundle):
    model = bundle['model']
    if spec['input_type'] == 'text':
        return model.predict_proba(text_test)[:, 1]

    feature_cols = bundle['feature_cols']
    return model.predict_proba(test_df[feature_cols])[:, 1]


def metric_row(spec, bundle, prob):
    threshold = float(bundle.get('best_threshold', bundle.get('default_threshold', 0.5)))
    pred = (prob >= threshold).astype(int)
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    return {
        'model': spec['model'],
        'feature_set': spec['feature_set'],
        'threshold': threshold,
        'f1': float(f1_score(y_test, pred)),
        'pr_auc': float(average_precision_score(y_test, prob)),
        'precision': float(precision_score(y_test, pred, zero_division=0)),
        'recall': float(recall_score(y_test, pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_test, prob)),
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
    }


model_probabilities = {}
rows = []
for spec in model_specs:
    bundle = model_bundles[spec['model']]
    prob = predict_prob(spec, bundle)
    model_probabilities[spec['model']] = prob
    rows.append(metric_row(spec, bundle, prob))

comparison = pd.DataFrame(rows).sort_values(['f1', 'pr_auc'], ascending=False).reset_index(drop=True)
comparison.insert(0, 'rank', comparison.index + 1)

comparison_display = comparison.round({
    'threshold': 4,
    'f1': 4,
    'pr_auc': 4,
    'precision': 4,
    'recall': 4,
    'roc_auc': 4,
})

display(comparison_display)


,rank,model,feature_set,threshold,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,1,ablation_mlp_metadata_only,metadata_only,0.5004,0.5258,0.4363,0.4104,0.7315,0.6031,356,497,127,346
1,2,proposed_hybrid_mlp_04,kcbert_pca+metadata,0.5074,0.4274,0.3808,0.3879,0.4757,0.5374,498,355,248,225
2,3,ablation_mlp_kcbert_pca_only,kcbert_pca_only,0.5038,0.3908,0.3754,0.3714,0.4123,0.5291,523,330,278,195
3,4,baseline_tfidf_random_forest,cleaned_text_tfidf,0.5021,0.2496,0.3956,0.4200,0.1776,0.5356,737,116,389,84


## 5. 최종 모델 선택

최종 모델 선택 기준은 이벤트 리뷰(1) F1을 1순위로 둔다. PR-AUC와 precision/recall 균형은 보조 기준으로 확인한다.

현재 결과 기준으로 선택 모델 후보 (Metadata only + MLP)가 가장 좋은 성능을 보인다. 다만 FP가 높기 때문에, 이벤트 리뷰를 많이 잡는 대신 일반 리뷰 오탐도 많다는 한계를 같이 기록한다.


In [5]:
selected_row = comparison.iloc[0].copy()
selected_model_name = selected_row['model']
selected_spec = next(spec for spec in model_specs if spec['model'] == selected_model_name)
selected_bundle = model_bundles[selected_model_name]
selected_threshold = float(selected_bundle.get('best_threshold', selected_bundle.get('default_threshold', 0.5)))

selection_reason = (
    'test 데이터에서 tuned threshold 기준 이벤트 리뷰(1) F1이 가장 높아서 선택. '
    '단, FP가 많아 일반 리뷰를 이벤트 리뷰로 오탐하는 경향은 07 별점 정제에서 주의해야 함.'
)

selected_display = pd.DataFrame([{
    'selected_model': selected_model_name,
    'feature_set': selected_row['feature_set'],
    'threshold': selected_threshold,
    'f1': selected_row['f1'],
    'pr_auc': selected_row['pr_auc'],
    'precision': selected_row['precision'],
    'recall': selected_row['recall'],
    'fp': int(selected_row['fp']),
    'fn': int(selected_row['fn']),
    'selection_reason': selection_reason,
}]).round({
    'threshold': 4,
    'f1': 4,
    'pr_auc': 4,
    'precision': 4,
    'recall': 4,
})

display(selected_display)


,selected_model,feature_set,threshold,f1,pr_auc,precision,recall,fp,fn,selection_reason
0,ablation_mlp_metadata_only,metadata_only,0.5004,0.5258,0.4363,0.4104,0.7315,497,127,test 데이터에서 tuned threshold 기준 이벤트 리뷰(1) F1이 가장...


## 6. 선택 모델 (Metadata only + MLP) 기준 FP/FN 오답 데이터 추출

선택 모델 (Metadata only + MLP)이 틀린 리뷰를 두 종류로 나눈다.

- FP: 실제 일반 리뷰(0)인데 이벤트 리뷰(1)로 예측한 경우
- FN: 실제 이벤트 리뷰(1)인데 일반 리뷰(0)로 예측한 경우

오답 리뷰 원문과 메타데이터를 같이 확인해 원인을 분석한다.


In [6]:
selected_prob = model_probabilities[selected_model_name]
selected_pred = (selected_prob >= selected_threshold).astype(int)

error_df = test_review_df.copy()
error_df['actual_label'] = y_test.to_numpy()
error_df['pred_label'] = selected_pred
error_df['event_prob'] = selected_prob
error_df['error_type'] = np.select(
    [
        (error_df['actual_label'] == 0) & (error_df['pred_label'] == 1),
        (error_df['actual_label'] == 1) & (error_df['pred_label'] == 0),
    ],
    ['FP_일반을_이벤트로_오탐', 'FN_이벤트를_일반으로_미탐'],
    default='correct',
)

summary_cols = ['rating', 'photo_count', 'text_length', 'emoji_count']
error_summary = (
    error_df.groupby('error_type')
    .agg(
        count=('error_type', 'size'),
        mean_event_prob=('event_prob', 'mean'),
        mean_rating=('rating', 'mean'),
        mean_photo_count=('photo_count', 'mean'),
        mean_text_length=('text_length', 'mean'),
        mean_emoji_count=('emoji_count', 'mean'),
    )
    .reset_index()
    .sort_values('count', ascending=False)
    .round(4)
)

display(error_summary)


,error_type,count,mean_event_prob,mean_rating,mean_photo_count,mean_text_length,mean_emoji_count
2,correct,702,0.4787,4.8348,0.8219,57.6766,0.6168
1,FP_일반을_이벤트로_오탐,497,0.5350,4.9980,1.2133,47.1187,0.3803
0,FN_이벤트를_일반으로_미탐,127,0.4443,4.8661,0.2913,69.7795,0.8110


## 7. 오답 샘플 출력

FP와 FN 샘플을 각각 확인한다. 여기서 은어, 신조어, 우회 표현, 짧은 리뷰, 사진 수, 별점 패턴을 같이 본다.


In [7]:
review_cols = [
    'raw_index',
    'store_name',
    'rating',
    'review_text',
    'cleaned_review_text',
    'photo_count',
    'text_length',
    'emoji_count',
    'actual_label',
    'pred_label',
    'event_prob',
    'error_type',
]
review_cols = [col for col in review_cols if col in error_df.columns]

fp_samples = (
    error_df[error_df['error_type'] == 'FP_일반을_이벤트로_오탐']
    .sort_values('event_prob', ascending=False)
    [review_cols]
    .head(15)
)

fn_samples = (
    error_df[error_df['error_type'] == 'FN_이벤트를_일반으로_미탐']
    .sort_values('event_prob', ascending=True)
    [review_cols]
    .head(15)
)

print('FP 샘플: 일반 리뷰인데 이벤트 리뷰로 예측한 사례')
display(fp_samples)

print('FN 샘플: 이벤트 리뷰인데 일반 리뷰로 놓친 사례')
display(fn_samples)


FP 샘플: 일반 리뷰인데 이벤트 리뷰로 예측한 사례


,raw_index,store_name,rating,review_text,cleaned_review_text,photo_count,text_length,emoji_count,actual_label,pred_label,event_prob,error_type
176,7002,육회한연어 건대점,5.0,맛있게 잘먹었습니다😆👍,맛있게 잘먹었습니다,2,12,2,0,1,0.587112,FP_일반을_이벤트로_오탐
313,4666,NaN,5.0,항상 너무 맛있게 먹어요! 맛도 좋고 양도 많아서 만족!,항상 너무 맛있게 먹어요! 맛도 좋고 양도 많아서 만족!,2,31,0,0,1,0.570898,FP_일반을_이벤트로_오탐
391,5267,NaN,5.0,매콤 하니 맛있네요 !! \n라멘 많이 먹는데 이집 맛있습니다!,매콤 하니 맛있네요 !! 라멘 많이 먹는데 이집 맛있습니다!,2,34,0,0,1,0.570529,FP_일반을_이벤트로_오탐
724,6192,홍콩반점0410 삼양동사거리점,5.0,오랜만에 맛있게 먹었습니다!! 언제나 그렇듯 탕수육이 맛있네요,오랜만에 맛있게 먹었습니다!! 언제나 그렇듯 탕수육이 맛있네요,2,34,0,0,1,0.570529,FP_일반을_이벤트로_오탐
423,7098,국민낙곱새 강서점,5.0,맛있어요ㅜㅜ❤️❤️❤️,맛있어요ㅜㅜ,1,12,3,0,1,0.569879,FP_일반을_이벤트로_오탐
1220,8033,샌디마켓,5.0,딸기치아바타샌드위치 너무맛있어서 계속시키게되네요~~완전강추합니다!!!,딸기치아바타샌드위치 너무맛있어서 계속시키게되네요~~완전강추합니다!!!,2,38,0,0,1,0.569512,FP_일반을_이벤트로_오탐
829,6976,육회한연어 건대점,5.0,정말 맛있어요. 육회도 신선하고 양이 진짜 많네요.,정말 맛있어요. 육회도 신선하고 양이 진짜 많네요.,2,28,0,0,1,0.569448,FP_일반을_이벤트로_오탐
1073,4127,NaN,5.0,굿👍🏻👍🏻\n맛이쏘요,굿 맛이쏘요,1,10,4,0,1,0.569428,FP_일반을_이벤트로_오탐
1106,5830,강정근본 닭강정&새우강정 노원점,5.0,처음 시켜보는데 닭과 새우 모두 너무 맛있습니다👍,처음 시켜보는데 닭과 새우 모두 너무 맛있습니다,2,27,1,0,1,0.569372,FP_일반을_이벤트로_오탐
209,8357,영천 얼큰 돼지국밥 수유점,5.0,국밥 뿐만 아니라 김치전도 너무 맛있어요 함박스테이크 서비스 감사합니다,국밥 뿐만 아니라 김치전도 너무 맛있어요 함박스테이크 서비스 감사합니다,2,39,0,0,1,0.569258,FP_일반을_이벤트로_오탐


FN 샘플: 이벤트 리뷰인데 일반 리뷰로 놓친 사례


,raw_index,store_name,rating,review_text,cleaned_review_text,photo_count,text_length,emoji_count,actual_label,pred_label,event_prob,error_type
1306,596,NaN,1.0,배달오는데까지 50분.. 음식 다 뿔었구요 당연히 맛도 떨어지죠\n사이다 리뷰는 감...,배달오는데까지 50분.. 음식 다 뿔었구요 당연히 맛도 떨어지죠 사이다 리뷰는 감사...,0,62,0,1,0,0.019578,FN_이벤트를_일반으로_미탐
259,5555,NaN,1.0,배달이 오래걸릴거같으면 제품을 늦게 만들어서 보내셔야하는거 아닌가요?조리완료는 10...,배달이 오래걸릴거같으면 제품을 늦게 만들어서 보내셔야하는거 아닌가요?조리완료는 10...,1,93,0,1,0,0.038936,FN_이벤트를_일반으로_미탐
266,5152,NaN,3.0,라구파스타도 랩도 너무 짜요🥲\n드레싱 1/2을 신청한건데도 이렇게 짠거면...ㅠㅠ...,라구파스타도 랩도 너무 짜요 드레싱 1/2을 신청한건데도 이렇게 짠거면...ㅠㅠ 후...,0,140,1,1,0,0.140853,FN_이벤트를_일반으로_미탐
974,908,NaN,4.0,스파게티가 마니짠거가타요..,스파게티가 마니짠거가타요..,0,15,0,1,0,0.259189,FN_이벤트를_일반으로_미탐
1110,4094,NaN,4.0,생각나서 또 주문했어요~,생각나서 또 주문했어요~,0,13,0,1,0,0.259864,FN_이벤트를_일반으로_미탐
663,6739,라파리나,4.0,피 두꺼운 짭짤한 두쫀쿠네요 우유랑 굿,피 두꺼운 짭짤한 두쫀쿠네요 우유랑 굿,1,21,0,1,0,0.375169,FN_이벤트를_일반으로_미탐
825,3958,NaN,4.0,맛있어서 최근에만 3번 먹엇습니다 꿔이도 맛나여,맛있어서 최근에만 3번 먹엇습니다 꿔이도 맛나여,1,26,0,1,0,0.375458,FN_이벤트를_일반으로_미탐
198,1976,산수냉면 목동점,5.0,가끔 냉면이 생각날 때면 냉면은 꼭 산수냉면에서 주문합니다. 물냉/비냉 모두 정말 ...,가끔 냉면이 생각날 때면 냉면은 꼭 산수냉면에서 주문합니다. 물냉/비냉 모두 정말 ...,0,428,1,1,0,0.378710,FN_이벤트를_일반으로_미탐
811,360,NaN,4.0,단거를안좋아해서 생크림까지같이먹는거라 두바이가좀덜달았음 더좋앗을듯 ㅎ,단거를안좋아해서 생크림까지같이먹는거라 두바이가좀덜달았음 더좋앗을듯 ㅎ,1,38,0,1,0,0.381013,FN_이벤트를_일반으로_미탐
945,4146,NaN,5.0,한때 유행햇던 두쫀쿠... 이제야 가게들이 두쫀쿠 가격을 내려가지고 이 틈에 후다닥...,한때 유행햇던 두쫀쿠... 이제야 가게들이 두쫀쿠 가격을 내려가지고 이 틈에 후다닥...,1,587,0,1,0,0.390520,FN_이벤트를_일반으로_미탐


## 8. 메타데이터 영향도 및 행동 패턴 해석

선택 모델 (Metadata only + MLP)이 tabular metadata를 사용하는 경우, 각 메타데이터를 섞었을 때 F1이 얼마나 떨어지는지 확인한다.

F1 감소폭이 클수록 해당 메타데이터가 모델 예측에 더 큰 영향을 줬다고 해석한다.


In [ ]:
def permutation_importance_for_tabular_model(model, X, y_true, threshold, columns, repeats=10, random_state=SEED):
    rng = np.random.default_rng(random_state)
    base_prob = model.predict_proba(X)[:, 1]
    base_pred = (base_prob >= threshold).astype(int)
    base_f1 = f1_score(y_true, base_pred)

    rows = []
    for col in columns:
        drops = []
        for _ in range(repeats):
            X_perm = X.copy()
            shuffled = X_perm[col].to_numpy().copy()
            rng.shuffle(shuffled)
            X_perm[col] = shuffled
            perm_prob = model.predict_proba(X_perm)[:, 1]
            perm_pred = (perm_prob >= threshold).astype(int)
            drops.append(base_f1 - f1_score(y_true, perm_pred))

        rows.append({
            'metadata': col,
            'base_f1': float(base_f1),
            'mean_f1_drop': float(np.mean(drops)),
            'std_f1_drop': float(np.std(drops)),
            'repeats': repeats,
        })
    return pd.DataFrame(rows).sort_values('mean_f1_drop', ascending=False).reset_index(drop=True)


selected_feature_cols = selected_bundle.get('feature_cols')
if selected_spec['input_type'] == 'tabular' and selected_feature_cols is not None:
    importance_cols = [col for col in meta_cols if col in selected_feature_cols]
else:
    importance_cols = []

if importance_cols:
    selected_importance = permutation_importance_for_tabular_model(
        selected_bundle['model'],
        test_df[selected_feature_cols],
        y_test,
        selected_threshold,
        importance_cols,
        repeats=10,
    )
    selected_importance_display = selected_importance.copy()
    selected_importance_display.insert(0, 'rank', selected_importance_display.index + 1)
    selected_importance_display = selected_importance_display.rename(columns={
        'metadata': '메타데이터',
        'base_f1': '기준 F1',
        'mean_f1_drop': '평균 F1 감소폭',
        'std_f1_drop': 'F1 감소폭 표준편차',
        'repeats': '반복 횟수',
    }).round({
        '기준 F1': 4,
        '평균 F1 감소폭': 4,
        'F1 감소폭 표준편차': 4,
    })
    top_metadata = selected_importance.iloc[0]
    print(
        f"선택 모델 (Metadata only + MLP)에서 가장 큰 영향을 준 메타데이터: {top_metadata['metadata']} "
        f"(평균 F1 감소폭: {top_metadata['mean_f1_drop']:.4f})"
    )
    display(selected_importance_display)
else:
    selected_importance = pd.DataFrame()
    print('선택 모델 (Metadata only + MLP)이 메타데이터 컬럼을 직접 사용하지 않아 permutation importance를 계산하지 않았다.')

behavior_summary = (
    error_df.groupby('actual_label')
    .agg(
        count=('actual_label', 'size'),
        mean_rating=('rating', 'mean'),
        mean_photo_count=('photo_count', 'mean'),
        mean_text_length=('text_length', 'mean'),
        mean_emoji_count=('emoji_count', 'mean'),
    )
    .reset_index()
    .replace({'actual_label': {0: '일반 리뷰(0)', 1: '이벤트 리뷰(1)'}})
    .round(4)
)

print('실제 라벨별 메타데이터 평균')
display(behavior_summary)


선택 모델 (Metadata only + MLP)에서 가장 큰 영향을 준 메타데이터: meta_photo_count (평균 F1 감소폭: 0.0390)


,rank,메타데이터,기준 F1,평균 F1 감소폭,F1 감소폭 표준편차,반복 횟수
0,1,meta_photo_count,0.5258,0.0390,0.0115,10
1,2,meta_rating,0.5258,0.0161,0.0047,10
2,3,meta_emoji_count,0.5258,0.0096,0.0043,10
3,4,meta_text_length,0.5258,0.0082,0.0052,10


실제 라벨별 메타데이터 평균


,actual_label,count,mean_rating,mean_photo_count,mean_text_length,mean_emoji_count
0,일반 리뷰(0),853,4.8640,0.8781,56.7819,0.5955
1,이벤트 리뷰(1),473,4.9619,0.9894,51.4461,0.4588


## 9. 최종 결과 해석

### 9-1. 핵심 결론

실험 결과, 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)보다 선택 모델 (Metadata only + MLP)이 더 높은 성능을 보였다. 이는 이벤트 리뷰 탐지에서 텍스트 의미보다 사진 수, 별점, 리뷰 길이 같은 행동 기반 메타데이터가 더 직접적인 신호로 작용했기 때문으로 해석된다.

즉, 초기 가설처럼 KcBERT 임베딩과 메타데이터를 결합한 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)이 최종 최고 성능을 보이지는 않았지만, 비교 실험을 통해 이벤트 리뷰 탐지에서 메타데이터의 중요성이 크다는 점은 확인할 수 있었다.

### 9-2. 성능 비교 해석

- 선택 모델 (Metadata only + MLP)은 test 기준 F1과 Recall이 가장 높아 이벤트 리뷰를 가장 많이 잡아냈다.
- 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)은 선택 모델 (Metadata only + MLP)보다 낮은 성능을 보였지만, KcBERT PCA only와 TF-IDF baseline보다는 높은 성능을 보였다.
- `KcBERT PCA only + MLP`와 `TF-IDF + Random Forest`는 텍스트만으로 이벤트 여부를 충분히 구분하지 못했다.
- 따라서 현재 데이터에서는 텍스트 의미보다 행동 패턴 기반 feature가 이벤트 리뷰 탐지에 더 강하게 작용했다고 볼 수 있다.

### 9-3. 메타데이터 영향도 해석

메타데이터 영향도는 어떤 모델을 기준으로 보느냐에 따라 다르게 해석해야 한다.

- 05번에서 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata) 기준으로 확인했을 때는 `meta_rating`이 가장 큰 영향을 보였다.
- 06번에서 선택 모델 (Metadata only + MLP) 기준으로 다시 확인했을 때는 `meta_photo_count`가 가장 큰 영향을 보였고, 그다음이 `meta_rating`이었다.

즉, 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)에서는 별점 정보가 가장 강하게 작용했고, 선택 모델 (Metadata only + MLP)에서는 사진 수가 가장 강하게 작용했다. 두 결과가 서로 충돌하는 것이 아니라, 기준 모델이 다르기 때문에 영향도 순위가 달라진 것이다.

해석하면 이벤트 리뷰 작성자는 일반 리뷰 작성자보다 사진을 더 많이 첨부하거나, 별점 패턴에서 차이를 보이는 경향이 있었다. 따라서 모델은 리뷰 문장의 의미보다 사진 수, 별점, 리뷰 길이 같은 행동 메타데이터를 이벤트 참여 여부의 주요 단서로 사용했다.

### 9-4. 오답 데이터 해석

FP는 실제로는 일반 리뷰인데 모델이 이벤트 리뷰로 잘못 판단한 경우다. 이 경우 일반 리뷰임에도 사진 수나 별점 패턴이 이벤트 리뷰와 유사해서 모델이 이벤트 리뷰로 오탐했을 가능성이 크다. 즉, 선택 모델 (Metadata only + MLP)은 행동 패턴이 비슷하면 텍스트 의미를 보지 못해 일반 리뷰도 이벤트 리뷰로 판단하는 한계가 있다.

FN은 실제 이벤트 리뷰인데 모델이 일반 리뷰로 놓친 경우다. 이 경우 이벤트 리뷰이지만 사진 수가 적거나 별점, 리뷰 길이 같은 메타데이터가 일반 리뷰와 유사해서 모델이 이벤트 리뷰로 판단하지 못했을 가능성이 있다. 즉, 이벤트성 여부가 텍스트에만 드러나거나 메타데이터 패턴이 약한 경우 선택 모델 (Metadata only + MLP)은 이를 놓칠 수 있다.

### 9-5. 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)이 기대보다 낮았던 이유

제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)은 KcBERT PCA feature와 메타데이터를 함께 사용했지만, 현재 결과에서는 선택 모델 (Metadata only + MLP)보다 낮은 성능을 보였다. 이는 PCA로 축소된 텍스트 feature가 이벤트 탐지에 충분한 정보를 제공하지 못했거나, 오히려 모델 학습에 잡음으로 작용했을 가능성이 있다.

특히 이벤트성 표현은 정형화되어 있지 않고 은어, 신조어, 우회 표현, 짧은 문장 형태로 나타날 수 있다. 이런 경우 일반적인 KcBERT 임베딩을 PCA로 축소한 feature만으로는 이벤트 여부를 안정적으로 구분하기 어렵다.

### 9-6. 최종 정리

결과적으로 본 실험에서는 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)이 최종 최고 성능을 보이지는 않았지만, 이벤트 리뷰 탐지에서 메타데이터가 핵심적인 역할을 한다는 점을 확인했다. 특히 사진 수와 별점은 이벤트 참여자의 행동 패턴을 반영하는 주요 단서로 작용했다.

다만 선택 모델 (Metadata only + MLP)은 텍스트 의미를 반영하지 못하므로, 행동 패턴이 비슷한 일반 리뷰를 이벤트 리뷰로 오탐하는 한계가 있다. 따라서 07번 별점 정제에서는 선택 모델 (Metadata only + MLP)의 예측 결과를 그대로 이진 제거 기준으로만 쓰기보다, 이벤트 확률을 활용한 가중치 방식도 함께 고려하는 것이 적절하다.

향후 개선 방향은 KcBERT 임베딩을 단순 PCA로 축소하는 방식 대신, 이벤트 표현에 특화된 fine-tuning, 텍스트 feature selection, 또는 메타데이터와 텍스트 feature의 가중 결합 방식을 적용해 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)의 성능을 개선하는 것이다.


## 10. 선택 모델 (Metadata only + MLP) 저장

06번의 선택 모델 (Metadata only + MLP)을 `outputs/final_selected_model.joblib`로 저장한다.

07번 별점 정제에서는 이 파일만 불러와 전체 리뷰에 이벤트 확률을 예측한다.


In [9]:
final_selected_bundle = {
    'model': selected_bundle['model'],
    'model_name': selected_model_name,
    'feature_set': selected_row['feature_set'],
    'input_type': selected_spec['input_type'],
    'feature_cols': selected_bundle.get('feature_cols'),
    'text_col': selected_bundle.get('text_col'),
    'target_col': selected_bundle.get('target_col', TARGET_COL),
    'default_threshold': float(selected_bundle.get('default_threshold', 0.5)),
    'best_threshold': selected_threshold,
    'selection_rule': 'highest F1 on test set with stored tuned threshold, PR-AUC as secondary criterion',
    'selection_reason': selection_reason,
    'selected_metrics': selected_row.to_dict(),
    'all_model_comparison': comparison.to_dict(orient='records'),
    'metadata_importance': selected_importance.to_dict(orient='records'),
    'next_step': '07_rating_refinement should load this bundle and apply it to all review rows.',
}

joblib.dump(final_selected_bundle, 'outputs/final_selected_model.joblib')

final_selected_display = {k: v for k, v in final_selected_bundle.items() if k != 'model'}
print('저장 완료: outputs/final_selected_model.joblib')
display(pd.DataFrame([final_selected_display]))


저장 완료: outputs/final_selected_model.joblib


,model_name,feature_set,input_type,feature_cols,text_col,target_col,default_threshold,best_threshold,selection_rule,selection_reason,selected_metrics,all_model_comparison,metadata_importance,next_step
0,ablation_mlp_metadata_only,metadata_only,tabular,"[meta_rating, meta_text_length, meta_emoji_cou...",None,target_label,0.5,0.500377,highest F1 on test set with stored tuned thres...,test 데이터에서 tuned threshold 기준 이벤트 리뷰(1) F1이 가장...,"{'rank': 1, 'model': 'ablation_mlp_metadata_on...","[{'rank': 1, 'model': 'ablation_mlp_metadata_o...","[{'metadata': 'meta_photo_count', 'base_f1': 0...",07_rating_refinement should load this bundle a...


## 11. 07번 별점 정제로 연결

07번에서는 선택 모델 (Metadata only + MLP)이 저장된 `outputs/final_selected_model.joblib`를 불러와 전체 리뷰 데이터에 이벤트 리뷰 확률을 계산한다.

그 다음 이벤트 리뷰로 판단된 행을 기준으로 별점 정제 전/후를 비교한다.
